# Build an SRE Incident Response Agent with Claude Managed Agents

## Introduction

When a production alert fires at 3 a.m., someone has to pull the logs, find the right runbook, trace the misconfiguration, open a PR, and get it approved. An agent can take that first pass for you and have a fix waiting for review by the time you're at the keyboard — as long as it has the right context and a human makes the final call.

[Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview) gives you the scalable infrastructure, sandboxing, & security pieces to build that with ease. In this tutorial you'll wire them together:

- A simulated **PagerDuty webhook** triggers your Claude Managed Agent with one API call.
- A **Skill** teaches the agent your team's runbook conventions, so it knows where to look.
- The built-in `bash`/`read`/`edit` tools let it investigate logs and infrastructure code in a sandbox.
- **Custom tools** let it open a pull request and ask a human to approve before merging — your code handles those calls, so you decide what "open a PR" actually does.
- The **Anthropic Console** records every step automatically, providing you complete observability.

Everything below runs with only `ANTHROPIC_API_KEY`. PagerDuty, GitHub, and Datadog are mocked with local fixtures so you can focus on the Managed Agents pieces; the closing section shows how to swap each mock for the real service.

### What you'll learn

- Upload a Skill and attach it to a Claude Managed Agent
- Mix the built-in toolset with custom tools your application handles
- Start a session from a webhook payload
- Gate a destructive action behind human approval
- Read the full session trace in the Console

### Prerequisites

Set `ANTHROPIC_API_KEY` in your environment, then install dependencies:

In [1]:
%pip install -q "anthropic>=0.91.0" python-dotenv

In [2]:
import hashlib
import json
import os
import time
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv
from utilities import wait_for_idle_status

load_dotenv()
client = Anthropic()
MODEL = os.getenv("COOKBOOK_MODEL", "claude-opus-4-6")
FIXTURE = Path("example_data/sre")

## 1. Upload a runbook skill

A [**Skill**](https://platform.claude.com/docs/en/managed-agents/skills) is a small filesystem bundle the platform mounts into the agent's context with progressive disclosure: the agent sees a one-line description up front and reads the body only when it's relevant. It's a good place for team conventions that shouldn't live in the system prompt.

The sample skill below encodes one rule — *consult the runbook before touching infrastructure* — the way a real team playbook would. You upload it once via the Skills API and reference it by ID on every agent that needs it.

In [3]:
# A real skill is usually a folder on disk (SKILL.md plus any helper
# scripts or reference docs) that you zip and upload. For this tutorial
# the SKILL.md is small enough to keep inline.
RUNBOOK_SKILL = """\
---
name: incident-runbooks
description: How to triage production incidents using the team runbooks.
---

# Incident runbooks

When an alert references a service, locate that service's recent logs
and identify the failure signature (the repeating error class, exit
code, or status pattern).

Consult the team runbooks before proposing any fix. Runbooks are
organised by failure signature — for example `oom.md`, `5xx.md`,
`latency.md`. Each one lists the triage steps for that class of
failure and the configuration that usually needs to change.

Any fix to infrastructure code must be opened as a pull request that
cites the runbook you followed. Do not patch live resources directly.
"""

skill = client.beta.skills.create(
    display_title="incident-runbooks",
    files=[("incident-runbooks/SKILL.md", RUNBOOK_SKILL.encode(), "text/markdown")],
)
print(f"skill: {skill.id} (version {skill.latest_version})")

skill: skill_01WPWHALbtEVBUWG6mHa7Tna (version 1775588716519983)


## 2. Create the agent

The agent's `tools` list combines three kinds of capability:

- [`agent_toolset_20260401`](https://platform.claude.com/docs/en/managed-agents/tools) — the built-in `bash`, `read`, `grep`, `edit`, … tools that run *inside* the sandbox. The agent uses these to investigate.
- The runbook **skill** from step 1.
- Two **custom tools** — `open_pull_request` and `request_approval` — that the agent can call but *your application* executes. The merge credential stays outside the agent's tool set, so the application executor owns the irreversible step.

The system prompt is persona and workflow only. The alert itself arrives as the first user event, so the same agent handles any incident.

In [4]:
SRE_SYSTEM_PROMPT = """\
You are an on-call SRE agent. Each user message is a PagerDuty alert
payload. Triage it to root cause and ship the minimal safe fix.

The session workspace contains the recent logs, the infrastructure
repo, and the team runbooks for the alerting service. Explore it to
find what you need.

Workflow for every alert:
1. Read the logs and identify the failure signature.
2. Find the root cause in the infrastructure repo, save a copy of the
   original file, edit it in place, then produce a unified diff with
   `diff -u`.
3. open_pull_request(title, body, diff) with the fix.
4. Use the returned pr_number and action_digest in
   request_approval(pr_number, action_digest, summary).
5. Wait for the application executor to approve or reject the exact
   action. Report its result; do not attempt the merge yourself.

Keep the fix minimal — do not refactor unrelated config.
"""

agent = client.beta.agents.create(
    name="cookbook-sre-responder",
    model=MODEL,
    system=SRE_SYSTEM_PROMPT,
    skills=[{"type": "custom", "skill_id": skill.id, "version": skill.latest_version}],
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        },
        {
            "type": "custom",
            "name": "open_pull_request",
            "description": "Open a pull request against the infra repo with the proposed fix.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "body": {"type": "string"},
                    "diff": {"type": "string", "description": "Unified diff of the change."},
                },
                "required": ["title", "body", "diff"],
            },
        },
        {
            "type": "custom",
            "name": "request_approval",
            "description": "Ask the on-call human to approve the proposed PR before merging.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "pr_number": {"type": "integer"},
                    "action_digest": {"type": "string"},
                    "summary": {"type": "string"},
                },
                "required": ["pr_number", "action_digest", "summary"],
            },
        },
    ],
)
print(f"agent: {agent.id} v{agent.version}")

agent: agent_011CZpw3Y76Vu4t2j2QEosVa v1


## 3. Create an environment and mount the data

The agent needs three things in its workspace to investigate: the recent service logs, the infrastructure repo, and the team runbooks. Upload each via the Files API and list them as `resources` so they're mounted into every session at the paths the system prompt expects. A `limited`-networking cloud environment is enough because the agent only needs its own filesystem.

To keep this notebook runnable with only `ANTHROPIC_API_KEY`, the infra "repo" is a single manifest with a too-low `memory: 128Mi` limit. In production you'd replace that upload with a `github_repository` resource that clones the real repo straight into the sandbox:

```python
{
    "type": "github_repository",
    "url": "https://github.com/your-org/infra",
    "authorization_token": os.environ["GITHUB_TOKEN"],
    "checkout": {"type": "branch", "name": "main"},
    "mount_path": "infra",
}
```

In [5]:
env = client.beta.environments.create(
    name="cookbook-sre-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)


def upload(path: Path, mime: str) -> str:
    with path.open("rb") as f:
        return client.beta.files.upload(file=(path.name, f, mime)).id


log_id = upload(FIXTURE / "logs/checkout-svc.log", "text/plain")
manifest_id = upload(FIXTURE / "infra/k8s/checkout-deploy.yaml", "text/yaml")
runbook_id = upload(FIXTURE / "runbooks/oom.md", "text/markdown")

RESOURCES = [
    {"type": "file", "file_id": log_id, "mount_path": "logs/checkout-svc.log"},
    {"type": "file", "file_id": manifest_id, "mount_path": "infra/k8s/checkout-deploy.yaml"},
    {"type": "file", "file_id": runbook_id, "mount_path": "runbooks/oom.md"},
]
print(f"environment: {env.id}")

environment: env_01R6hmJkd6BhpPotXnoC7rqU


## 4. Handle the incident alert

The handler below is the one function you'd deploy — a Flask or FastAPI route that your alerting system calls when an incident fires. It creates a session referencing the agent and environment, mounts the data, and sends the alert JSON as the first `user.message` event. This example uses a [PagerDuty V3 webhook](https://developer.pagerduty.com/docs/webhooks-overview) payload, but any pager that can POST JSON works the same way; here you call the handler directly with the fixture.

In [6]:
def handle_pagerduty_webhook(payload: dict) -> str:
    incident = payload["event"]["data"]
    session = client.beta.sessions.create(
        environment_id=env.id,
        agent={"type": "agent", "id": agent.id, "version": agent.version},
        resources=RESOURCES,
        title=f"[{incident['service']['summary']}] {incident['title']}",
    )
    client.beta.sessions.events.send(
        session.id,
        events=[
            {
                "type": "user.message",
                "content": [{"type": "text", "text": json.dumps(payload, indent=2)}],
            }
        ],
    )
    return session.id


with (FIXTURE / "alert.json").open() as f:
    alert = json.load(f)

session_id = handle_pagerduty_webhook(alert)
print(f"session: {session_id}")

session: sesn_011CZpw3gtC691y7qmaLNmLM


## 5. Service the agent's custom tool calls

This is where the built-in tools and your custom tools come together. The agent's `read`/`bash`/`edit` calls run on the container and appear in the event log as `agent.tool_use` — that's the investigation, and you just print it. But when the agent calls one of your custom tools, the session goes `idle` with `stop_reason.type == "requires_action"` and waits for *your application* to respond with a `user.custom_tool_result`.

The loop below polls `events.list` and answers `open_pull_request` by writing to a local list — that's the GitHub mock — but **returns** when `request_approval` arrives. The application records the exact PR digest before presenting the decision to a human. The agent never receives a merge tool.

In production, "needs a human" usually means *post it to Slack*: drop the agent's summary into the on-call channel with an **Approve** button, and send the result back when someone clicks. The [`slack_data_bot` cookbook](slack_data_bot.ipynb) shows the Bolt wiring for that; here you'll approve inline in the next cell so the notebook stays self-contained.

In [7]:
prs: list[dict] = []
pending_approvals: dict[str, dict] = {}
action_reservations: dict[str, str] = {}
seen_events: set[str] = set()
DELIVERY_SLA_SECONDS = 90


def merge_action_digest(pr: dict) -> str:
    action = {
        "action": "github.pull_request.merge",
        "pr_number": pr["number"],
        "title": pr["title"],
        "body": pr["body"],
        "diff": pr["diff"],
    }
    canonical = json.dumps(action, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode()).hexdigest()


def handle_custom_tool(name: str, args: dict) -> dict:
    if name == "open_pull_request":
        n = len(prs) + 1
        pr = {"number": n, "merged": False, **args}
        pr["action_digest"] = merge_action_digest(pr)
        prs.append(pr)
        print(f"\n── PR #{n}: {args['title']} ──")
        return {
            "pr_number": n,
            "url": f"mock://infra/pull/{n}",
            "action_digest": pr["action_digest"],
        }
    raise ValueError(f"unhandled tool {name}")


def release_action_reservation(approval: dict, now: float | None = None) -> bool:
    """Release only the reservation still owned by this terminal request."""
    action_digest = approval["action_digest"]
    if action_reservations.get(action_digest) != approval["event_id"]:
        return False
    del action_reservations[action_digest]
    approval["reservation_released_at"] = time.time() if now is None else now
    return True


def register_approval(
    event_id: str,
    pr: dict,
    summary: str,
    ttl: float = 900,
    delivery_sla: float = DELIVERY_SLA_SECONDS,
) -> dict:
    """Reserve one approval request for one exact action digest."""
    action_digest = merge_action_digest(pr)
    existing_event_id = action_reservations.get(action_digest)
    if existing_event_id is not None:
        return {
            "status": "refused",
            "reason": "action_already_reserved",
            "existing_event_id": existing_event_id,
        }
    requested_at = time.time()
    pending_approvals[event_id] = {
        "event_id": event_id,
        "pr_number": pr["number"],
        "action_digest": action_digest,
        "summary": summary,
        "requested_at": requested_at,
        "delivery_deadline": requested_at + delivery_sla,
        "delivered_at": None,
        "answered_at": None,
        "expires_at": requested_at + ttl,
        "consumed": False,
        "outcome": None,
        "reported_outcome": None,
        "reservation_released_at": None,
    }
    action_reservations[action_digest] = event_id
    return {"status": "pending", "event_id": event_id, "action_digest": action_digest}


def mark_approval_delivered(event_id: str, delivered_at: float | None = None) -> dict:
    """Record delivery only after the external channel reports success."""
    approval = pending_approvals.get(event_id)
    if approval is None:
        return {"status": "refused", "reason": "unknown_approval"}
    if approval["delivered_at"] is not None:
        return {"status": "delivered", "event_id": event_id}
    if approval["outcome"] is not None:
        return {"status": "refused", "reason": "approval_terminal"}
    delivered_at = time.time() if delivered_at is None else delivered_at
    if delivered_at > approval["delivery_deadline"]:
        approval["consumed"] = True
        approval["outcome"] = "approval_never_delivered"
        release_action_reservation(approval, delivered_at)
        return {"status": "refused", "reason": "delivery_sla_exceeded"}
    approval["delivered_at"] = delivered_at
    return {"status": "delivered", "event_id": event_id}


def stale_approvals(now: float | None = None) -> list[dict]:
    """Transition and report each undelivered or expired request exactly once."""
    now = time.time() if now is None else now
    incidents = []
    for event_id, approval in pending_approvals.items():
        if (
            approval["outcome"] is None
            and approval["delivered_at"] is None
            and now > approval["delivery_deadline"]
        ):
            approval["consumed"] = True
            approval["outcome"] = "approval_never_delivered"
            release_action_reservation(approval, now)
        elif approval["outcome"] is None and now > approval["expires_at"]:
            approval["consumed"] = True
            approval["outcome"] = "expired_undecided"
            release_action_reservation(approval, now)
        if approval["outcome"] not in {
            "approval_never_delivered",
            "expired_undecided",
            "expired_clicked_late",
        }:
            continue
        if approval["reported_outcome"] == approval["outcome"]:
            continue
        approval["reported_outcome"] = approval["outcome"]
        incident = {
            "event_id": event_id,
            "pr_number": approval["pr_number"],
            "status": "attention_required",
            "reason": (
                approval["outcome"]
                if approval["outcome"].startswith("approval_")
                else f"approval_{approval['outcome']}"
            ),
            "age_seconds": round(now - approval["requested_at"], 1),
        }
        if approval["outcome"] == "approval_never_delivered":
            incident["delivery_late_by_seconds"] = round(now - approval["delivery_deadline"], 1)
        else:
            incident["expired_by_seconds"] = round(now - approval["expires_at"], 1)
        incidents.append(incident)
    return incidents


def consume_approval(event_id: str, decision: str) -> dict:
    approval = pending_approvals.get(event_id)
    if approval is None:
        return {"status": "refused", "reason": "unknown_approval"}
    if approval["outcome"] == "approval_never_delivered":
        return {"status": "refused", "reason": "approval_never_delivered"}
    if approval["outcome"] in {"expired_undecided", "expired_clicked_late"}:
        return {"status": "refused", "reason": "expired"}
    if approval["consumed"]:
        return {"status": "refused", "reason": "already_consumed"}
    if approval["delivered_at"] is None:
        return {"status": "refused", "reason": "approval_not_delivered"}
    now = time.time()
    approval["consumed"] = True  # reserve before the side effect or rejection
    approval["answered_at"] = now
    if decision != "approved":
        approval["outcome"] = "human_rejected"
        return {"status": "rejected", "reason": "human_rejected"}
    if now > approval["expires_at"]:
        approval["outcome"] = "expired_clicked_late"
        release_action_reservation(approval, now)
        return {"status": "refused", "reason": "expired"}
    if action_reservations.get(approval["action_digest"]) != event_id:
        approval["outcome"] = "action_reservation_lost"
        return {"status": "refused", "reason": "action_already_reserved"}
    pr = prs[approval["pr_number"] - 1]
    if merge_action_digest(pr) != approval["action_digest"]:
        approval["outcome"] = "action_changed"
        release_action_reservation(approval, now)
        return {"status": "refused", "reason": "action_changed"}
    if pr["merged"]:
        approval["outcome"] = "action_already_executed"
        return {"status": "refused", "reason": "action_already_executed"}
    pr["merged"] = True  # the application executor performs the effect
    approval["outcome"] = "executed"
    return {
        "status": "executed",
        "receipt_id": event_id,
        "pr_number": pr["number"],
        "action_digest": approval["action_digest"],
    }


def run_until_approval_or_end(session_id: str) -> str | None:
    """Poll the session's event log, servicing custom tools, until either
    a request_approval call arrives (return its event_id so the caller
    can respond) or the agent ends its turn (return None)."""
    custom_calls: dict[str, object] = {}
    responded: set[str] = set()
    while True:
        idle_stop = None
        for ev in client.beta.sessions.events.list(session_id):
            if ev.id in seen_events:
                continue
            seen_events.add(ev.id)
            if ev.type == "agent.message":
                for block in ev.content:
                    if block.type == "text":
                        print(block.text, end="")
            elif ev.type == "agent.tool_use":
                print(f"\n  [{ev.name}]")
            elif ev.type == "agent.custom_tool_use":
                custom_calls[ev.id] = ev
                print(f"\n→ {ev.name}")
            elif ev.type == "session.status_idle":
                idle_stop = ev.stop_reason
            elif ev.type == "session.status_terminated":
                return None
        if idle_stop is None:
            time.sleep(1.0)
            continue
        if idle_stop.type == "end_turn":
            return None
        if idle_stop.type == "requires_action":
            for event_id in idle_stop.event_ids:
                if event_id in responded:
                    continue
                call = custom_calls[event_id]
                if call.name == "request_approval":
                    pr = prs[call.input["pr_number"] - 1]
                    if call.input["action_digest"] != merge_action_digest(pr):
                        result = {"status": "refused", "reason": "action_digest_mismatch"}
                        client.beta.sessions.events.send(
                            session_id,
                            events=[
                                {
                                    "type": "user.custom_tool_result",
                                    "custom_tool_use_id": event_id,
                                    "content": [{"type": "text", "text": json.dumps(result)}],
                                }
                            ],
                        )
                        responded.add(event_id)
                        continue
                    registration = register_approval(event_id, pr, call.input["summary"])
                    if registration["status"] != "pending":
                        client.beta.sessions.events.send(
                            session_id,
                            events=[
                                {
                                    "type": "user.custom_tool_result",
                                    "custom_tool_use_id": event_id,
                                    "content": [{"type": "text", "text": json.dumps(registration)}],
                                }
                            ],
                        )
                        responded.add(event_id)
                        continue
                    return event_id
                result = handle_custom_tool(call.name, call.input)
                client.beta.sessions.events.send(
                    session_id,
                    events=[
                        {
                            "type": "user.custom_tool_result",
                            "custom_tool_use_id": event_id,
                            "content": [{"type": "text", "text": json.dumps(result)}],
                        }
                    ],
                )
                responded.add(event_id)
        time.sleep(1.0)


approval_event_id = run_until_approval_or_end(session_id)

I'll help you triage this production incident. The alert shows checkout-svc pods are crash-looping with 7 restarts in 5 minutes. Let me start by exploring the workspace to find logs and the infrastructure configuration.
  [bash]
Let me check the session directory for logs and infrastructure files:
  [bash]
Perfect! Let me examine the logs first to identify the failure signature:
  [read]
Perfect! I've identified the failure signature. The logs show a clear **OutOfMemoryError** pattern:

**Failure Signature:**
- Service starts and warms pricing cache (14,092 entries)
- Heap memory grows rapidly: 101MB → 118MB → 121MB (against 128MB limit)
- `pricing.recompute` function attempts allocation during garbage collection pause (412ms GC pause observed)
- Container gets OOMKilled (exit 137) after ~2 minutes
- Service restarts and immediately repeats the cycle

Now let me examine the deployment configuration to identify the root cause:
  [read]
**Root Cause Analysis:**

The deployment has memory

The agent has read the logs, matched the `OOMKilled` signature to `runbooks/oom.md` via the skill, found the 128Mi memory limit in `infra/k8s/checkout-deploy.yaml`, edited it, opened a PR, and is now waiting on you. This is the message that would land in your `#oncall` Slack channel:

In [8]:
pr = prs[0]
print(pr["body"], "\n")
print(pr["diff"])
print("\n" + "─" * 60)
print("APPROVAL REQUESTED:", pending_approvals[approval_event_id]["summary"])
delivery = mark_approval_delivered(approval_event_id)
assert delivery["status"] == "delivered"

## Issue
checkout-svc pods are in a CrashLoopBackOff state due to OutOfMemoryError. The service consistently crashes after ~2 minutes with 7 restarts in the last 5 minutes.

## Root Cause
The deployment had memory limits set to 128Mi, which is insufficient for the pricing cache operation:
- Pricing cache warms with 14,092 entries during startup
- Heap pressure builds to 118-121MB (92-94% of limit) within 90 seconds
- pricing.recompute fails to allocate 8MB, causing OOMKilled (exit 137)
- Service restarts and repeats the cycle

## Fix
Increase memory allocation to provide adequate headroom:
- **Memory request:** 128Mi → 256Mi
- **Memory limit:** 128Mi → 512Mi

This provides 4x headroom for the pricing cache and normal operations while remaining resource-efficient (512Mi limit is standard for Java/similar workloads with caching).

## Verification
The fix addresses the immediate OOMKilled pattern in logs and aligns memory resources with the actual cache size and operational requirements. 

### Exercise the executor boundary

These key-free checks run the approval logic directly. They demonstrate action-level duplicate refusal, token replay refusal, approve-A/execute-B refusal, delivery failure, expiry, rejection followed by a second approval, and an unanswered request. Delivery has a 90-second service-level deadline separate from the 15-minute human decision window. Undelivered and expired requests become terminal, alert once, and release their action reservation so a new request can be created; the old token stays dead. A human rejection deliberately keeps the action reserved so an agent cannot keep asking until someone says yes.

In [9]:
def demo_request(
    label: str,
    diff: str,
    ttl: float = 900,
    delivery_sla: float = DELIVERY_SLA_SECONDS,
    delivered: bool = True,
) -> tuple[str, dict]:
    pr = {
        "number": len(prs) + 1,
        "merged": False,
        "title": f"demo: {label}",
        "body": "executor-bound approval check",
        "diff": diff,
    }
    pr["action_digest"] = merge_action_digest(pr)
    prs.append(pr)
    event_id = f"demo-{label}"
    registration = register_approval(event_id, pr, label, ttl, delivery_sla)
    assert registration["status"] == "pending"
    if delivered:
        delivery = mark_approval_delivered(event_id)
        assert delivery["status"] == "delivered"
    return event_id, pr


results = {}

replay_event, replay_pr = demo_request("replay", "memory: 512Mi")
results["first_approval"] = consume_approval(replay_event, "approved")
results["replay"] = consume_approval(replay_event, "approved")
assert results["first_approval"]["status"] == "executed"
assert results["replay"] == {"status": "refused", "reason": "already_consumed"}

reserved_event, reserved_pr = demo_request("action-reservation", "memory: 768Mi")
results["duplicate_action_request"] = register_approval(
    "demo-action-reservation-retry", reserved_pr, "same exact merge, second token"
)
results["reserved_action_approval"] = consume_approval(reserved_event, "approved")
assert results["duplicate_action_request"] == {
    "status": "refused",
    "reason": "action_already_reserved",
    "existing_event_id": reserved_event,
}
assert results["reserved_action_approval"]["status"] == "executed"
assert reserved_pr["merged"] is True

mutation_event, mutation_pr = demo_request("mutation", "memory: 512Mi")
mutation_pr["diff"] = "memory: 512Mi\nreplicas: 0"
results["approve_A_execute_B"] = consume_approval(mutation_event, "approved")
assert results["approve_A_execute_B"] == {"status": "refused", "reason": "action_changed"}
assert mutation_pr["merged"] is False

expiry_event, expiry_pr = demo_request("expiry", "memory: 512Mi", ttl=-1)
results["late_approval"] = consume_approval(expiry_event, "approved")
results["late_click_visibility"] = next(
    item for item in stale_approvals() if item["event_id"] == expiry_event
)
assert results["late_approval"] == {"status": "refused", "reason": "expired"}
assert results["late_click_visibility"]["reason"] == "approval_expired_clicked_late"
assert all(item["event_id"] != expiry_event for item in stale_approvals())
results["approval_after_expiry"] = register_approval(
    "demo-expiry-retry", expiry_pr, "fresh approval after terminal expiry"
)
assert results["approval_after_expiry"]["status"] == "pending"
assert mark_approval_delivered("demo-expiry-retry")["status"] == "delivered"
results["reauthorized_after_expiry"] = consume_approval("demo-expiry-retry", "approved")
assert results["reauthorized_after_expiry"]["status"] == "executed"
assert expiry_pr["merged"] is True

reject_event, reject_pr = demo_request("rejection", "memory: 512Mi")
results["rejection"] = consume_approval(reject_event, "rejected")
results["approval_after_rejection"] = consume_approval(reject_event, "approved")
results["new_request_after_rejection"] = register_approval(
    "demo-rejection-retry", reject_pr, "same action after a human said no"
)
assert results["rejection"] == {"status": "rejected", "reason": "human_rejected"}
assert results["approval_after_rejection"] == {"status": "refused", "reason": "already_consumed"}
assert results["new_request_after_rejection"] == {
    "status": "refused",
    "reason": "action_already_reserved",
    "existing_event_id": reject_event,
}
assert reject_pr["merged"] is False

undelivered_event, undelivered_pr = demo_request(
    "undelivered",
    "memory: 640Mi",
    delivery_sla=90,
    delivered=False,
)
undelivered_now = pending_approvals[undelivered_event]["delivery_deadline"] + 1
results["registered_but_never_delivered"] = next(
    item for item in stale_approvals(now=undelivered_now) if item["event_id"] == undelivered_event
)
results["undelivered_token_stays_dead"] = consume_approval(undelivered_event, "approved")
results["fresh_request_after_delivery_failure"] = register_approval(
    "demo-undelivered-retry",
    undelivered_pr,
    "fresh approval after delivery failure",
)
assert results["registered_but_never_delivered"]["reason"] == ("approval_never_delivered")
assert results["undelivered_token_stays_dead"] == {
    "status": "refused",
    "reason": "approval_never_delivered",
}
assert results["fresh_request_after_delivery_failure"]["status"] == "pending"
assert mark_approval_delivered("demo-undelivered-retry")["status"] == "delivered"
results["reauthorized_after_delivery_failure"] = consume_approval(
    "demo-undelivered-retry", "approved"
)
assert results["reauthorized_after_delivery_failure"]["status"] == "executed"
assert all(item["event_id"] != undelivered_event for item in stale_approvals(now=undelivered_now))
assert undelivered_pr["merged"] is True

silent_event, silent_pr = demo_request("silence", "memory: 512Mi", ttl=-1)
results["unanswered"] = next(item for item in stale_approvals() if item["event_id"] == silent_event)
assert results["unanswered"]["reason"] == "approval_expired_undecided"
assert all(item["event_id"] != silent_event for item in stale_approvals())
assert silent_pr["merged"] is False

for name, result in results.items():
    detail = f" {result['reason']}" if result.get("reason") else ""
    print(f"{name}: {result['status']}{detail}")

first_approval: executed
replay: refused already_consumed
duplicate_action_request: refused action_already_reserved
reserved_action_approval: executed
approve_A_execute_B: refused action_changed
late_approval: refused expired
late_click_visibility: attention_required approval_expired_clicked_late
approval_after_expiry: pending
reauthorized_after_expiry: executed
rejection: rejected human_rejected
approval_after_rejection: refused already_consumed
new_request_after_rejection: refused action_already_reserved
registered_but_never_delivered: attention_required approval_never_delivered
undelivered_token_stays_dead: refused approval_never_delivered
fresh_request_after_delivery_failure: pending
reauthorized_after_delivery_failure: executed
unanswered: attention_required approval_expired_undecided


## 6. Approve and let the executor merge

The application reserves the exact PR digest when it registers the first approval request, so a retry cannot create a second token for the same merge. It records delivery only after the external channel succeeds, then consumes the approval and performs the merge itself. Only then does it return the execution receipt to the agent. Undelivered and expired requests terminate the old token and release the reservation for a fresh request. A human rejection keeps the action reserved by policy, so an agent cannot keep asking until someone says yes. In the Slack version this runs across the message sender and button-click handler. The merge credential never enters the agent's tool set.

In [10]:
approval_receipt = consume_approval(approval_event_id, decision="approved")
client.beta.sessions.events.send(
    session_id,
    events=[
        {
            "type": "user.custom_tool_result",
            "custom_tool_use_id": approval_event_id,
            "content": [{"type": "text", "text": json.dumps(approval_receipt)}],
        }
    ],
)

run_until_approval_or_end(session_id)
print(f"\n\nReceipt: {approval_receipt['receipt_id']}")
print(f"PR #{pr['number']} merged: {prs[0]['merged']}")

The executor verified the approval receipt and merged the exact proposed PR.
## ✅ Incident Resolved

**Summary:**
- **Status:** MERGED (PR #1)
- **Failure:** checkout-svc crash-loop with OOMKilled (exit 137)
- **Root Cause:** Memory limit of 128Mi was insufficient for pricing cache (14,092 entries)
- **Fix Applied:** 
  - Memory request: 128Mi → 256Mi
  - Memory limit: 128Mi → 512Mi

**What Happened:**
1. Service loaded 14k+ pricing cache entries during startup
2. Heap grew to 118-121MB within ~2 minutes (92-94% of 128Mi limit)
3. pricing.recompute failed to allocate 8MB, triggering OutOfMemoryError
4. Container was OOMKilled and restarted, repeating the cycle

**Expected Outcome:**
With 512Mi limit and 256Mi request, the service will have sufficient memory headroom for:
- Pricing cache operations
- Normal request processing
- JVM garbage collection pauses
- No more CrashLoopBackOff

The deployment update will trigger a rolling restart of the 3 replicas, allowing pods to spawn with the

## 7. Review the run in the Console

The important boundary is outside the prompt: Claude can propose the PR and request approval, but it cannot call the merge operation. The application reserves the exact PR digest when the request is registered, applies a 90-second delivery deadline and a separate 15-minute human window, consumes the approval once, performs the merge, and returns a receipt. It refuses duplicate requests for the same action, approve-A/execute-B, expired approvals, token replay, and resurrection after rejection. `stale_approvals()` preserves and reports non-delivery, unanswered expiry, and late-click expiry as separate terminal outcomes, once each; none can trigger the merge. Expired and undelivered requests release only their own reservation after becoming terminal. The `action_reservation_lost` check remains defense-in-depth for a future durable-store race; the single-process harness does not claim to exercise it.

This pattern incorporates the production failure report shared by Anton Dziatkovskii and Mycroft in [issue #701](https://github.com/anthropics/claude-cookbooks/issues/701) and the fleet-derived negative cases they ran in [PR #803](https://github.com/anthropics/claude-cookbooks/pull/803#issuecomment-5160512049), together with the action-bound admission and consumption work contributed by [EMILIA Protocol](https://github.com/emiliaprotocol/emilia-protocol/pull/451). The collaboration is concrete: Mycroft supplies incident-shaped failure cases; EMILIA turns them into executor invariants and receipts.

Because the investigation ran as a Managed Agents session, the file reads, `bash` diff, manifest edit, proposal and approval request are persisted in the Console. The application-owned receipt records the final merge boundary separately. Open the session under **Managed Agents → Sessions** to review both sides of the handoff:

<img src="https://raw.githubusercontent.com/anthropics/claude-cookbooks/main/managed_agents/example_data/sre/console_session.png" alt="Console session view for the incident-response run" width="700" />

### Cleanup

Archive the session and the resources you created.

In [11]:
wait_for_idle_status(client, session_id)
client.beta.sessions.archive(session_id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
client.beta.skills.versions.delete(skill.latest_version, skill_id=skill.id)
client.beta.skills.delete(skill.id)
print("archived")

archived


## Next steps: production wiring

Three swaps take this from notebook to on-call.

**Approve in Slack.** When `request_approval` arrives, post it to the on-call channel with Block Kit buttons and send the `user.custom_tool_result` back from the action handler. The [`slack_data_bot` cookbook](slack_data_bot.ipynb) covers the Bolt app setup; the approval-specific bit is small:

```python
def post_for_approval(session_id, event_id, summary):
    response = slack.client.chat_postMessage(
        channel=ONCALL_CHANNEL,
        text=summary,
        blocks=[
            {"type": "section", "text": {"type": "mrkdwn", "text": summary}},
            {"type": "actions", "elements": [
                {"type": "button", "text": {"type": "plain_text", "text": "Approve"},
                 "action_id": "approve", "value": f"{session_id}:{event_id}"},
                {"type": "button", "text": {"type": "plain_text", "text": "Reject"},
                 "action_id": "reject", "value": f"{session_id}:{event_id}"},
            ]},
        ],
    )
    delivery = mark_approval_delivered(event_id)
    if delivery["status"] != "delivered":
        raise RuntimeError(f"approval delivery was not admitted: {delivery}")
    return response

@slack.action("approve")
def on_approve(ack, body):
    ack()
    session_id, event_id = body["actions"][0]["value"].split(":")
    approval_receipt = consume_approval(event_id, decision="approved")
    client.beta.sessions.events.send(
        session_id,
        events=[{"type": "user.custom_tool_result",
                 "custom_tool_use_id": event_id,
                 "content": [{"type": "text",
                              "text": json.dumps(approval_receipt)}]}],
    )
```

In production, persist the same distinct timestamps used here: registered, delivery deadline, delivered, answered, reservation released, plus terminal outcome and last-reported outcome. Reserve `action_digest` atomically in the durable store (for example with a unique constraint); the notebook dictionary is only a single-process mock. Write `delivered_at` only after the channel reports success. Alert once when a request misses its delivery deadline, expires without an answer, or receives a late click. Undelivered and expired requests may release their reservation only after the old token is terminal; creating the replacement still requires a fresh approval request. A human rejection deliberately keeps its action reserved until an explicit operator re-arm. None of these liveness paths retries the merge.

To drop the polling loop entirely, register a Console webhook on `session.requires_action` — the platform calls your endpoint the moment the agent pauses, and you post to Slack from there.

**GitHub instead of the mock.** Give the agent a GitHub credential that can read and open pull requests, but cannot merge them. Keep the merge credential in the application executor that consumes the approval. [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb) walks through per-user credentials.

```python
agent = client.beta.agents.create(
    ...,
    mcp_servers=[{"type": "url", "name": "github",
                  "url": "https://api.githubcopilot.com/mcp/"}],
    tools=[{"type": "agent_toolset_20260401", ...},
           {"type": "mcp_toolset", "server_name": "github"},  # no merge scope
           {"type": "custom", "name": "request_approval", ...}],
)
session = client.beta.sessions.create(..., vault_ids=[github_vault.id])
```

**Live logs instead of a fixture.** Pass `DD_API_KEY` / `DD_APP_KEY` through the environment config and let the agent `curl` the Datadog Logs API from `bash` instead of reading a mounted file.

> See also the [Agent SDK site-reliability agent](https://github.com/anthropics/claude-cookbooks/blob/main/claude_agent_sdk/03_The_site_reliability_agent.ipynb) for the same problem solved with the local Agent SDK instead of the hosted Managed Agents runtime.


## What you learned

- Trigger a session from any external event — one API call from a PagerDuty webhook started the whole run.
- Attach a **Skill** to give the agent your team's conventions.
- Mount data with **resources**: `github_repository` for code, `file` for logs and runbooks.
- Use **custom tools** to call back into your app and gate actions on human approval via `requires_action`.
- Bind approval to the exact action, reserve that action across duplicate requests, expire it, consume it once, and keep the irreversible credential in the application executor.
- Track registration, delivery, decision, and terminal outcome separately so silence and late clicks remain visible without repeated alerts or weakened fail-closed behavior.
- Join the Console session trace with the application-owned execution receipt for an end-to-end audit trail.

Swap the mocks for GitHub MCP, a Slack approval button, and live logs, and it's ready for on-call.